In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
SRC  = ROOT / "src"
DATA = ROOT / "data"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from companies import HOLDINGS_INFO, RUSSELL_SECTOR_WEIGHTS
from portfolio_optimizer_new import (
    OptimizerBands,
    compute_alpha_vector,
    optimize_quadratic_portfolio,
)

In [2]:
tickers = sorted(HOLDINGS_INFO.keys())
print("Universe size:", len(tickers))

universe_sectors = sorted({HOLDINGS_INFO[t]["sector"] for t in tickers})
russell_sectors = sorted(RUSSELL_SECTOR_WEIGHTS.keys())

missing = [s for s in russell_sectors if s not in universe_sectors]
print("Russell sectors not in universe:", missing)


Universe size: 26
Russell sectors not in universe: ['Communications', 'Consumer Staples']


In [3]:
alpha = compute_alpha_vector(tickers=tickers, holdings_info=HOLDINGS_INFO)

assert alpha.shape == (len(tickers),)
assert np.isfinite(alpha).all()

# ETFs alpha should be 0 unless you add expense_ratio fields
etfs = [t for t in tickers if HOLDINGS_INFO[t]["asset_type"] == "index"]
stocks = [t for t in tickers if HOLDINGS_INFO[t]["asset_type"] != "index"]

print("Num ETFs:", len(etfs), "Num stocks:", len(stocks))
print("Alpha min/max:", float(alpha.min()), float(alpha.max()))

# Spot-check: stock alpha = er_stock - er_sector_etf
etf_er_by_sector = {
    HOLDINGS_INFO[t]["sector"]: float(HOLDINGS_INFO[t].get("expected_return", 0.0))
    for t in etfs
}
alpha_map = dict(zip(tickers, alpha))

for t in stocks[:8]:
    sec = HOLDINGS_INFO[t]["sector"]
    expected = float(HOLDINGS_INFO[t]["expected_return"]) - float(etf_er_by_sector.get(sec, 0.0))
    assert abs(alpha_map[t] - expected) < 1e-12, f"Alpha mismatch for {t}"

print("✅ compute_alpha_vector checks passed")


Num ETFs: 9 Num stocks: 17
Alpha min/max: -0.0055 0.24999999999999997
✅ compute_alpha_vector checks passed


In [4]:
prices = pd.read_csv(DATA / "daily_prices.csv", parse_dates=["date"]).set_index("date").sort_index()
missing_cols = [t for t in tickers if t not in prices.columns]
print("Missing price columns:", missing_cols)
assert not missing_cols

px = prices[tickers].ffill()
rets = px.pct_change().dropna()

print("Returns shape:", rets.shape)
print("Any NaNs in returns:", bool(rets.isna().any().any()))
assert not rets.isna().any().any()

print("✅ price/returns checks passed")


Missing price columns: []
Returns shape: (276, 26)
Any NaNs in returns: False
✅ price/returns checks passed


In [8]:
bands = OptimizerBands(
    etf_min=0.0, etf_max=0.1,
    stock_min=0.0, stock_max=1.0,
    max_weight_etf=0.15,
    max_weight_stock=0.15,
    sector_penalty_gamma=5.0,   # try 2, 5, 10
)

res = optimize_quadratic_portfolio(
    daily_prices_csv=DATA / "daily_prices.csv",
    holdings_info=HOLDINGS_INFO,
    russell_sector_weights=RUSSELL_SECTOR_WEIGHTS,
    te_cap=0.5,                # try 0.08 / 0.10 / 0.12 / 0.15
    bands=bands,
    lookback_days=252,
)

assert res is not None, "❌ Optimizer returned None (still infeasible or solver issue)."

w = pd.Series(res["weights"]).sort_values(ascending=False)
print("Solved. TE:", res["te"], "ActiveRet:", res["active_return"], "IR:", res["ir"])
print("\nTop weights:")
display(w.head(12))

# Basic feasibility checks
assert abs(w.sum() - 1.0) < 1e-6
assert (w >= -1e-10).all()
assert res["te"] <= 0.12 + 1e-6

# Sleeve checks
is_etf = pd.Series({t: HOLDINGS_INFO[t]["asset_type"] == "index" for t in w.index})
w_etf = float(w[is_etf].sum())
w_stock = float(w[~is_etf].sum())

print("\nETF weight:", w_etf, "Stock weight:", w_stock)
assert w_etf >= bands.etf_min - 1e-6 and w_etf <= bands.etf_max + 1e-6
assert w_stock >= bands.stock_min - 1e-6 and w_stock <= bands.stock_max + 1e-6

print("✅ optimize_quadratic_portfolio smoke test passed")


AssertionError: ❌ Optimizer returned None (still infeasible or solver issue).

In [ ]:
bands_bad = OptimizerBands(
    etf_min=0.90, etf_max=0.95,   # impossible with per-ETF caps and only 9 ETFs
    stock_min=0.10, stock_max=0.10,
    max_weight_etf=0.10,
    max_weight_stock=0.02,
    sector_penalty_gamma=5.0,
)

res_bad = optimize_quadratic_portfolio(
    daily_prices_csv=DATA / "daily_prices.csv",
    holdings_info=HOLDINGS_INFO,
    russell_sector_weights=RUSSELL_SECTOR_WEIGHTS,
    te_cap=0.12,
    bands=bands_bad,
)

print("Result (expected None):", res_bad)
assert res_bad is None
print("✅ infeasibility test passed")


Result (expected None): None
✅ infeasibility test passed
